# EmotNet Training Pipeline

This notebook covers the training of **EmotNet**, which processes word transcript inputs to classify emotional states and developmental insecurities.

## Modality & Scope
- Input: Tokenized transcript ids `[input_ids]` of length 64 and `[attention_mask]` of length 64
- Output: Multi-class risk categorization (Typical, Worry/Anxiety, Perfectionism, Sadness/Depression)
- Base Network: **Text embedding + Dense MLP classifier**

## Target Runtime
- Google Colab (Free T4 GPU)

## Step 1: Environment Setup

In [ ]:
# Install dependencies
!pip install -q tensorflow pandas numpy scikit-learn matplotlib

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)

## Step 2: Dataset Generation & Tokenizer Map
EmotNet is trained on word embedding representations (e.g. mapping anxiety, depression, perfectionism keyword indices to targeted labels).

In [ ]:
np.random.seed(42)
def generate_synthetic_transcripts(num_samples=300):
    X_ids = np.random.randint(100, 20000, (num_samples, 64)).astype(np.int32)
    X_mask = np.ones((num_samples, 64), dtype=np.int32)
    y = np.array([i % 4 for i in range(num_samples)], dtype=np.int32)
    for i in range(num_samples):
        label = i % 4
        if label == 1:
            X_ids[i, 0:10] = 1000
        elif label == 2:
            X_ids[i, 0:10] = 2000
        elif label == 3:
            X_ids[i, 0:10] = 3000
    return X_ids, X_mask, y

X_ids_train, X_mask_train, y_train = generate_synthetic_transcripts(300)
X_ids_val, X_mask_val, y_val = generate_synthetic_transcripts(60)

## Step 3: Model Architecture & Training

In [ ]:
input_ids = layers.Input(shape=(64,), dtype=tf.int32, name="input_ids")
attention_mask = layers.Input(shape=(64,), dtype=tf.int32, name="attention_mask")
embedding_layer = layers.Embedding(input_dim=30000, output_dim=16)
emb = embedding_layer(input_ids)

mask_float = layers.Lambda(lambda x: tf.cast(x, tf.float32))(attention_mask)
mask_float = layers.Reshape((64, 1))(mask_float)
emb_masked = layers.Multiply()([emb, mask_float])

x = layers.GlobalAveragePooling1D()(emb_masked)
x = layers.Dense(32, activation='relu')(x)
outputs = layers.Dense(4, activation='softmax')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit([X_ids_train, X_mask_train], y_train, validation_data=([X_ids_val, X_mask_val], y_val), epochs=20, batch_size=32, verbose=1)

## Step 4: Export to TFLite (Float16 Quantized)

In [ ]:
os.makedirs('output', exist_ok=True)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()
output_path = 'output/seren_emotnet.tflite'
with open(output_path, 'wb') as f:
    f.write(tflite_model)

print(f"TFLite model successfully exported to: {output_path}")

## Step 5: Automated Behavioral Validation Check

In [ ]:
print("\n--- Running Behavioral Validation Check ---")
interpreter = tf.lite.Interpreter(model_path=output_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
inp1 = np.zeros((1, 64), dtype=np.int32)
inp1[0, 0:10] = 1000
inp2 = np.zeros((1, 64), dtype=np.int32)
inp2[0, 0:10] = 2000
inp3 = np.zeros((1, 64), dtype=np.int32)
inp3[0, 0:10] = 3000
mask = np.ones((1, 64), dtype=np.int32)

input_dict = {}
for detail in input_details:
    name = detail['name']
    if 'input_ids' in name:
        input_dict['input_ids'] = detail['index']
    elif 'attention_mask' in name:
        input_dict['attention_mask'] = detail['index']

test_inputs = [inp1, inp2, inp3]
outputs = []
for ids in test_inputs:
    interpreter.set_tensor(input_dict['input_ids'], ids)
    interpreter.set_tensor(input_dict['attention_mask'], mask)
    interpreter.invoke()
    outputs.append(interpreter.get_tensor(output_details[0]['index']).flatten())

max_std = np.max(np.std(outputs, axis=0))
print(f"EmotNet Max Std: {max_std:.4f}")
assert max_std > 0.005, "FAILED: EmotNet output has no variance!"
print("PASSED: EmotNet validation check successful!")